# 

In [1]:
%matplotlib widget

In [2]:
%%capture
from ocelot import *
from ocelot.gui import *
import matplotlib.pyplot as plt
import numpy as np
from felarc.simple_tba import get_tba_dispersion_condition, get_quad_strengths, TBADefinition, make_tba, TripleBendAchromat
from felarc.plot import plot_optics, plot_r56_full
from felarc.t20 import ARC_ANGLE_RAD
from felarc.maps import sequence_to_map, OpticalParameters, thin_quadrupole, drift, sbendas

The T20 arc comprises a $>6°$ bend.  We start with the simplest isochronous lattice: three identical dipoles, with a total bending angle equal to the 6° we are after.  We also choose a 3m dipole on the basis that in Nina's design we covered the $6°$ in 9m of dipole.

In [3]:
DIPOLE_ANGLE = ARC_ANGLE_RAD / 3
DIPOLE_LENGTH = 3
dx, dxp = get_tba_dispersion_condition(dipole_length=DIPOLE_LENGTH, dipole_angle=DIPOLE_ANGLE)

In [4]:
print(dx, dxp)

-0.0261740907458337 -0.01746038085732892


In [5]:
ad = TBADefinition(dipole_length=DIPOLE_LENGTH, dipole_angle=DIPOLE_ANGLE, l1=1., l2=1, l3=2., qlength=1e-6, k11=1, k12=-1)
tba = make_tba(ad)
k11, k12 = get_quad_strengths(ad)

Optimization terminated successfully.
         Current function value: 0.000000
         Iterations: 268
         Function evaluations: 507


In [6]:
start_index = 0
# stop_index = start_index+2
stop_index = tba.mlat.sequence.index(tba.markers.middle) + 1
tbamap = sequence_to_map(tba.mlat.sequence[start_index:stop_index], 14, variables=[tba.components.q1, tba.components.q2])

In [7]:
target_optics = OpticalParameters(beta_x=10, alpha_x=0, beta_y=10, alpha_y=0, dx=0, dy=0)
input_optics = OpticalParameters(beta_x=10, alpha_x=1, beta_y=20, alpha_y=-1, dx=0, dy=0)

In [8]:
output_twiss = tbamap * input_optics

In [9]:
repr(output_twiss.alpha_x.expand())

'-462.10061816445*K_q1**2*K_q2**2 + 1195.78035001106*K_q1**2*K_q2 - 773.616405429268*K_q1**2 + 1039.50335567576*K_q1*K_q2**2 - 1765.72740177837*K_q1*K_q2 + 544.483172344488*K_q1 - 577.355792371935*K_q2**2 + 454.523452221156*K_q2 - 83.7048237813247'

In [10]:
output_twiss.alpha_y.expand()

-1006.83734903652*K_q1**2*K_q2**2 + 2580.76767645741*K_q1**2*K_q2 - 1653.79880807772*K_q1**2 + 2264.89808020703*K_q1*K_q2**2 - 3791.80695341935*K_q1*K_q2 + 1139.48142738075*K_q1 - 1257.95844582871*K_q2**2 + 959.553712754394*K_q2 - 170.378808421726

In [11]:
from sympy.solvers import solve
solutions = solve([output_twiss.alpha_x, output_twiss.alpha_y], output_twiss.alpha_x.free_symbols, check=True)
solutions

[{K_q1: -2.52543925352702, K_q2: 1.01586791470777},
 {K_q1: 0.999593513232332, K_q2: -189051074016845.},
 {K_q1: 1.24992383172351, K_q2: -55242602476562.2},
 {K_q1: 4.77495659848321, K_q2: 1.56442840711635}]

In [18]:
print(solutions[0].items())
first, second = solutions[0].items()
print(first)
k1 = float(first[1])
k2 = float(second[2])
print(k1, k2)

dict_items([(K_q1, -2.52543925352702), (K_q2, 1.01586791470777)])
(K_q1, -2.52543925352702)


IndexError: tuple index out of range

In [18]:
print(solutions[0].items())
first, second = solutions[0].items()
print(first)
k1 = float(first[1])
k2 = float(second[2])
print(k1, k2)

dict_items([(K_q1, -2.52543925352702), (K_q2, 1.01586791470777)])
(K_q1, -2.52543925352702)


IndexError: tuple index out of range

In [15]:
thinning_factor = 1e6
# tba = make_tba(TBADefinition(dipole_length=DIPOLE_LENGTH, dipole_angle=DIPOLE_ANGLE, l1=1., l2=1, l3=2., qlength=1., k11=k1, k12=k2))
tba = make_tba(TBADefinition(dipole_length=DIPOLE_LENGTH, dipole_angle=DIPOLE_ANGLE, l1=1., l2=1, l3=2., qlength=1/thinning_factor, k11=k1*thinning_factor, k12=k2*thinning_factor))
# PASS

NameError: name 'k1' is not defined

In [ ]:
twiss0 = input_optics.to_twiss()
tba.mlat.sequence = tba.mlat.sequence[start_index: stop_index]
twisses = twiss(tba.mlat, tws0=twiss0, nPoints=1000)
plot_opt_func(tba.mlat, twisses)
print(twisses[-1])
np.set_printoptions(linewidth=250)

In [ ]:
d = drift(length=1.4, energy=14); d

In [ ]:
q = thin_quadrupole(k1l=k1); q

In [ ]:
Quadrupole(l=1e-6, k1=k1*1e6).R(energy=14)[0]

In [ ]:
sbend(length=3, angle=DIPOLE_ANGLE, energy=14)

In [ ]:
SBend(l=3, angle=DIPOLE_ANGLE).R(energy=10000)[1]

SBEND matches

drift, matches

thin quadrupole, matches

In [ ]:
print(k1)
(sbend(length=3, angle=DIPOLE_ANGLE, energy=14)
 * drift(length=1, energy=14)
 * thin_quadrupole(k1l=k1)
 # drift(length=1, energy=14)
 # * thin_quadrupole(k1l=k2)
)
# sbend(length=3, angle=DIPOLE_ANGLE, energy=10000) * drift(length=1, energy=14) 

In [ ]:
thin_quadrupole(k1l=k1).rmat

In [ ]:
MagneticLattice([SBend(l=3, angle=DIPOLE_ANGLE),
                 Drift(l=1),
                 Quadrupole(l=1e-6, k1=k1*1e6),
                 # Drift(l=1),
                 # Quadrupole(l=1e-6, k1=k2*1e6)
                ]).transfer_maps(energy=14)[1]
# MagneticLattice([SBend(l=3, angle=DIPOLE_ANGLE), Drift(l=1)]).transfer_maps(energy=14)[1]

In [ ]:
MagneticLattice([SBend(l=3, angle=DIPOLE_ANGLE),
                 Drift(l=1),
                 Quadrupole(l=1e-6, k1=k1*1e6),
                 # Drift(l=1),
                 # Quadrupole(l=1e-6, k1=k2*1e6)
                ]).transfer_maps(energy=14)[1]
# MagneticLattice([SBend(l=3, angle=DIPOLE_ANGLE), Drift(l=1)]).transfer_maps(energy=14)[1]

In [ ]:
MagneticLattice([SBend(l=3, angle=DIPOLE_ANGLE),
                 Drift(l=1),
                 Quadrupole(l=1e-6, k1=k1*1e6),
                 # Drift(l=1),
                 # Quadrupole(l=1e-6, k1=k2*1e6)
                ]).transfer_maps(energy=14)[1]
# MagneticLattice([SBend(l=3, angle=DIPOLE_ANGLE), Drift(l=1)]).transfer_maps(energy=14)[1]